# Cerrar el ASR semilla (sin seguir entrenando)

El entrenamiento de `train_asr.ipynb` se cortó en el último checkpoint. Este notebook **no entrena**:

1. Lee `trainer_state.json` del último checkpoint y toma `best_model_checkpoint` (el de menor WER en dev).
   Con `save_total_limit=2` ese checkpoint se conserva aunque el corte haya sido después.
2. Si no existe `modelo_final/`, lo crea a partir de ese checkpoint, junto con `historial.csv` y `curvas.png`.
3. Evalúa en dev, test_sk001 y test_otros (igual que la celda 9 de `train_asr`) y además calcula el
   **WER sin tildes**: quita las tildes de referencia y predicción antes de jiwer. No cambia el modelo.

Sale: `resultados.json`, `tabla_resultados.csv`, `pred_*.csv`, `curvas.png` en la misma carpeta `SALIDA`.

In [ ]:
#@title 1. Parámetros (los mismos de train_asr.ipynb)
DATASET_ID = ""  #@param {type:"string"}
DRIVE_DATASET = "/content/drive/MyDrive/TESIS/asr/dataset_v1.0.zip"  #@param {type:"string"}
SALIDA = "/content/drive/MyDrive/TESIS/asr/whisper_small_base"  #@param {type:"string"}

MODELO = "openai/whisper-small"
REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
IDIOMA = "spanish"
SEED = 1234
SHA_DATASET_PROYECTO = "174b61329123a19a4ed9d09b4d8cb229eb7989cddc9d63d9888859c906993a5d"

## 2. GPU y Drive

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)")
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")

## 3. Librerías (mismas versiones que el entrenamiento)

In [ ]:
!pip install -q transformers==5.17.0 accelerate==1.15.0 jiwer==4.0.0 gdown

## 4. Núcleo (idéntico al de `train_asr.ipynb`)

In [ ]:
%%writefile /content/asr_core.py
# ---- nucleo del ASR: se prueba localmente y se copia tal cual al notebook ----
import hashlib, re, unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

SR = 16000
FUENTE_SK = "menendez_gomez_2025"


# ============================ TEXTO ============================
def normalizar(t):
    """Misma normalizacion que build_dataset_v1.py (text_norm): NFC, minusculas, sin puntuacion.
    Se aplica tambien a lo que predice Whisper, que tiende a poner mayusculas y puntos."""
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[¡!¿?.,;:…\"“”«»()\[\]{}—–\-_/\\]", " ", t)
    t = t.replace("'", " ").replace("’", " ")
    return re.sub(r"\s+", " ", t).strip()


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


# ============================ SPLIT ============================
def hacer_split(meta, lineas_tts, seed, frac_dev, frac_test, frac_otros):
    """Particion fija para todos los experimentos de la tesis.

    - test_otros: grupos de hablante ENTEROS de youtube y workshop (voces que el modelo no oye).
    - dev / test_sk001: clips de SK001 cuyo texto NO aparece en el txt del TTS, para que los
      experimentos con audio sintetico no vean las frases de evaluacion.
    - Un texto de evaluacion nunca queda tambien en train (se saca de train: 'fuera_texto').
    """
    m = meta.sort_values("id").reset_index(drop=True).copy()
    m["split"] = "train"
    rng = np.random.RandomState(seed)

    # 1) hablantes no vistos
    for fuente in sorted(set(m.source) - {FUENTE_SK}):
        d = m[m.source == fuente].groupby("speaker_group").duration_s.sum()
        grupos = sorted(d.index)
        rng.shuffle(grupos)
        acum, elegidos = 0.0, []
        for g in grupos:
            if acum >= frac_otros * d.sum():
                break
            elegidos.append(g)
            acum += d[g]
        m.loc[m.speaker_group.isin(elegidos), "split"] = "test_otros"

    # 2) SK001: textos que no esten contenidos en ninguna linea del TTS
    blob = "\n".join(f" {normalizar(l)} " for l in lineas_tts if l.strip())
    sk = m[m.source == FUENTE_SK]
    elegibles = sorted(t for t in set(sk.text_norm) if f" {t} " not in blob)
    rng.shuffle(elegibles)
    n_test, n_dev = round(frac_test * len(sk)), round(frac_dev * len(sk))
    cuenta = sk.text_norm.value_counts()
    usados = {"test_sk001": 0, "dev": 0}
    for t in elegibles:
        destino = "test_sk001" if usados["test_sk001"] < n_test else "dev" if usados["dev"] < n_dev else None
        if destino is None:
            break
        m.loc[(m.source == FUENTE_SK) & (m.text_norm == t), "split"] = destino
        usados[destino] += int(cuenta[t])

    # 3) sin fuga por texto: evaluacion con texto en el TTS, o train con texto de evaluacion
    evaluacion = m.split != "train"
    en_tts = m.text_norm.map(lambda t: f" {t} " in blob)
    m.loc[evaluacion & en_tts, "split"] = "fuera_tts"
    textos_eval = set(m.loc[m.split.isin(["dev", "test_sk001", "test_otros"]), "text_norm"])
    m.loc[(m.split == "train") & m.text_norm.isin(textos_eval), "split"] = "fuera_texto"
    return m


def resumen_split(m):
    r = m.groupby("split").agg(clips=("id", "size"), minutos=("duration_s", lambda s: round(s.sum() / 60, 1)),
                               hablantes=("speaker_group", "nunique"))
    return r.reindex([s for s in ["train", "dev", "test_sk001", "test_otros", "fuera_tts", "fuera_texto"]
                      if s in r.index])


# ============================ DATOS PARA WHISPER ============================
class ClipsASR(torch.utils.data.Dataset):
    def __init__(self, df, raiz, processor):
        self.filas = list(df[["file_name", "text_norm"]].itertuples(index=False))
        self.raiz, self.p = Path(raiz), processor

    def __len__(self):
        return len(self.filas)

    def __getitem__(self, i):
        archivo, texto = self.filas[i]
        y, sr = sf.read(self.raiz / archivo, dtype="float32")
        assert sr == SR, f"{archivo}: {sr} Hz"
        return {"input_features": self.p.feature_extractor(y, sampling_rate=SR).input_features[0],
                "labels": self.p.tokenizer(texto).input_ids}


@dataclass
class Collator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all():   # el modelo lo agrega solo
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


# ============================ METRICAS ============================
def wer_cer(refs, hyps):
    import jiwer
    pares = [(r, h) for r, h in zip(refs, hyps) if r]            # jiwer no acepta referencias vacias
    r, h = [p[0] for p in pares], [p[1] for p in pares]
    return 100 * jiwer.wer(r, h), 100 * jiwer.cer(r, h)


def metricas_trainer(processor):
    tok = processor.tokenizer

    def f(pred):
        ids, lab = pred.predictions, pred.label_ids
        lab = np.where(lab == -100, tok.pad_token_id, lab)
        ids = np.where(ids == -100, tok.pad_token_id, ids)
        hyps = [normalizar(t) for t in tok.batch_decode(ids, skip_special_tokens=True)]
        refs = [normalizar(t) for t in tok.batch_decode(lab, skip_special_tokens=True)]
        w, c = wer_cer(refs, hyps)
        return {"wer": round(w, 2), "cer": round(c, 2)}
    return f


def wer_con_ic(pred, seed=0, n=1000):
    """WER corpus (no promedio por clip) e IC 95% por bootstrap sobre clips."""
    import jiwer
    p = pred[pred.text_norm.str.len() > 0]
    err, pal = [], []
    for a, b in zip(p.text_norm, p.prediccion):
        o = jiwer.process_words(a, b)
        err.append(o.substitutions + o.deletions + o.insertions)
        pal.append(o.substitutions + o.deletions + o.hits)
    err, pal = np.array(err), np.array(pal)
    rng = np.random.RandomState(seed)
    k = rng.randint(0, len(p), size=(n, len(p)))
    boot = 100 * err[k].sum(1) / pal[k].sum(1)
    return dict(wer=round(float(100 * err.sum() / pal.sum()), 2), ic95=[round(float(x), 2) for x in np.percentile(boot, [2.5, 97.5])],
                cer=round(100 * jiwer.cer(list(p.text_norm), list(p.prediccion)), 2), clips=int(len(p)),
                palabras=int(pal.sum()))


@torch.no_grad()
def transcribir(model, processor, df, raiz, device, batch=16, max_tokens=128, num_beams=1):
    """Transcribe df y devuelve una copia con la prediccion y el error por clip."""
    import jiwer
    model.eval()
    dtype = next(model.parameters()).dtype
    salida = []
    for i in range(0, len(df), batch):
        trozo = df.iloc[i:i + batch]
        audios = [sf.read(Path(raiz) / f, dtype="float32")[0] for f in trozo.file_name]
        x = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
        ids = model.generate(x.to(device, dtype), language="spanish", task="transcribe",
                             max_new_tokens=max_tokens, num_beams=num_beams)
        salida += processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    r = df[["id", "file_name", "source", "speaker_group", "duration_s", "text_norm"]].copy()
    r["prediccion"] = [normalizar(t) for t in salida]
    r["wer_clip"] = [round(100 * jiwer.wer(a, b), 1) if a else None for a, b in zip(r.text_norm, r.prediccion)]
    return r

## 5. Datos y partición guardada

Usa `split_v1.csv` tal cual quedó en Drive: no se recalcula.

In [ ]:
import sys, importlib, json, re, shutil, zipfile
sys.path.insert(0, "/content")
import asr_core as C
importlib.reload(C)
import pandas as pd
from pathlib import Path

out = Path(SALIDA)
if not out.is_dir():
    raise SystemExit(f"No existe {out}: revisa SALIDA en la celda 1 (debe ser la misma de train_asr).")


def id_de(v):
    v = v.strip()
    m = re.search(r"/d/([-\w]{20,})", v) or re.search(r"[?&]id=([-\w]{20,})", v)
    return m.group(1) if m else v


if DATASET_ID.strip():
    import gdown
    carpeta = Path("/content/descargas"); carpeta.mkdir(parents=True, exist_ok=True)
    ZIP = Path(gdown.download(id=id_de(DATASET_ID), output=str(carpeta) + "/", quiet=True) or "")
    if not ZIP.is_file():
        raise SystemExit("No pude descargar el dataset: compártelo como 'Cualquier persona con el enlace'.")
else:
    ZIP = Path(DRIVE_DATASET)
    if not ZIP.is_file():
        raise SystemExit(f"No encuentro {ZIP}. Pon DATASET_ID o corrige DRIVE_DATASET.")
sha_zip = C.sha256(ZIP)
print("dataset:", "= el del proyecto" if sha_zip == SHA_DATASET_PROYECTO else f"DISTINTO al del proyecto ({sha_zip[:12]})")

DATOS = Path("/content/datos") / sha_zip[:12]
if not DATOS.exists():
    tmp = DATOS.with_name(DATOS.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(tmp)
    tmp.rename(DATOS)
RAIZ = next(DATOS.rglob("metadata.csv")).parent
meta = pd.read_csv(RAIZ / "metadata.csv")

F_SPLIT = out.parent / "split_v1.csv"
if not F_SPLIT.exists():
    raise SystemExit(f"No encuentro {F_SPLIT}: es el que creó train_asr. Revisa SALIDA.")
part = pd.read_csv(F_SPLIT)[["id", "split"]]
split = meta.merge(part, on="id", how="inner")
if len(split) != len(part) or len(part) != len(meta):
    raise SystemExit(f"split_v1.csv ({len(part)}) no coincide con el dataset ({len(meta)}).")
print(C.resumen_split(split).to_string())

## 6. Mejor checkpoint

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

CK = out / "checkpoints"
ultimo = get_last_checkpoint(str(CK)) if CK.is_dir() else None
if ultimo is None:
    raise SystemExit(f"No hay checkpoints en {CK}")
estado = json.loads((Path(ultimo) / "trainer_state.json").read_text())
hist = pd.DataFrame(estado["log_history"])
evals = hist.dropna(subset=["eval_wer"])[["step", "eval_wer", "eval_cer", "eval_loss"]]
print(f"último checkpoint: {Path(ultimo).name} (paso {estado['global_step']} de {estado.get('max_steps')})")
print(evals.to_string(index=False))
print("\nexisten en Drive:", sorted(p.name for p in CK.glob("checkpoint-*")))

best = estado.get("best_model_checkpoint")
print(f"best_model_checkpoint: {best} | WER dev = {estado.get('best_metric')}")
# la ruta guardada es la de la sesión que entrenó: se busca por nombre dentro de CK
MEJOR = CK / Path(best).name if best else None
if MEJOR is None or not (MEJOR / "config.json").exists():
    wer_de = dict(zip(evals.step.astype(int), evals.eval_wer))
    existentes = [p for p in CK.glob("checkpoint-*") if int(p.name.split("-")[1]) in wer_de]
    MEJOR = min(existentes, key=lambda p: wer_de[int(p.name.split("-")[1])])
    print(f"AVISO: {best} no está; uso el de menor WER en dev entre los que existen: {MEJOR.name}")
PASO_MEJOR = int(MEJOR.name.split("-")[1])
print(f"\n==> se usa {MEJOR.name}")

## 7. `modelo_final`, historial y curvas

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import matplotlib.pyplot as plt

processor = WhisperProcessor.from_pretrained(MODELO, revision=REVISION, language=IDIOMA, task="transcribe")
FINAL = out / "modelo_final"
if (FINAL / "config.json").exists():
    print(f"Ya existe {FINAL}: se usa ese (no se sobrescribe).")
    model = WhisperForConditionalGeneration.from_pretrained(FINAL)
    o = FINAL / "origen.json"
    USADO = json.loads(o.read_text())["checkpoint"] if o.exists() else "modelo_final (train_asr)"
else:
    model = WhisperForConditionalGeneration.from_pretrained(MEJOR)
    tmp = FINAL.with_name("modelo_final.tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    model.save_pretrained(tmp)
    processor.save_pretrained(tmp)
    (tmp / "origen.json").write_text(json.dumps(
        {"checkpoint": MEJOR.name, "paso": PASO_MEJOR, "wer_dev_entrenamiento": estado.get("best_metric"),
         "entrenamiento_cortado_en": estado["global_step"]}, indent=2))
    tmp.rename(FINAL)                       # solo aparece si se guardó completo
    print(f"modelo_final creado desde {MEJOR.name}")
    USADO = MEJOR.name
model.generation_config.language = IDIOMA
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

hist.to_csv(out / "historial.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for col, nombre in (("loss", "train"), ("eval_loss", "dev")):
    if col in hist.columns:
        h = hist.dropna(subset=[col]); ax[0].plot(h.step, h[col], label=nombre)
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(evals.step, evals.eval_wer, marker="o")
ax[1].axvline(PASO_MEJOR, color="gray", ls="--", label=f"mejor ({PASO_MEJOR})"); ax[1].legend()
ax[1].set_title("WER dev (%)")
plt.tight_layout(); plt.savefig(out / "curvas.png", dpi=120); plt.show()

## 8. Evaluación final (= celda 9 de `train_asr`) + WER sin tildes

*Sin tildes* quita solo los acentos agudos y graves (á→a, é→e…), no la ñ ni la diéresis, en la referencia
y en la predicción. Mide cuánto del error es solo acentuación, que la ortografía del corpus no aplica de forma uniforme.

In [ ]:
import unicodedata

_ACENTOS = {"\u0301", "\u0300"}             # agudo y grave; la ñ (U+0303) se conserva


def sin_tildes(t):
    t = unicodedata.normalize("NFD", str(t))
    return unicodedata.normalize("NFC", "".join(ch for ch in t if ch not in _ACENTOS))


def metricas(pred):
    p2 = pred.assign(text_norm=pred.text_norm.map(sin_tildes), prediccion=pred.prediccion.map(sin_tildes))
    return {"con_tildes": C.wer_con_ic(pred), "sin_tildes": C.wer_con_ic(p2)}


device = torch.device("cuda")
model.to(device).eval()
PARTES = {s: split[split.split == s] for s in ("dev", "test_sk001", "test_otros")}
res = {"checkpoint": USADO}
for nombre, df in PARTES.items():
    pred = C.transcribir(model, processor, df, RAIZ, device)
    pred.to_csv(out / f"pred_{nombre}.csv", index=False)
    res[nombre] = metricas(pred)
    zs = out / f"pred_zero_shot_{nombre}.csv"
    if zs.exists():                         # zero-shot ya calculado en train_asr: solo se re-mide
        res[nombre]["zero_shot"] = metricas(pd.read_csv(zs, keep_default_na=False))
    print(nombre, "listo")
(out / "resultados.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

filas = []
for nombre in PARTES:
    r = res[nombre]
    fila = {"partición": nombre, "clips": r["con_tildes"]["clips"], "palabras": r["con_tildes"]["palabras"]}
    for k in ("con_tildes", "sin_tildes"):
        m = r[k]
        fila[f"WER {k.replace('_', ' ')}"] = f"{m['wer']} [{m['ic95'][0]}–{m['ic95'][1]}]"
        fila[f"CER {k.replace('_', ' ')}"] = m["cer"]
    fila["WER zero-shot"] = r.get("zero_shot", {}).get("con_tildes", {}).get("wer", "-")
    filas.append(fila)
tabla = pd.DataFrame(filas).set_index("partición")
tabla.to_csv(out / "tabla_resultados.csv")
print(f"\nModelo: {res['checkpoint']} (WER de corpus, IC 95% bootstrap sobre clips)\n")
print(tabla.to_string())

## 9. Tres ejemplos de voces no vistas

In [ ]:
import IPython.display as ipd
p = pd.read_csv(out / "pred_test_otros.csv", keep_default_na=False).sample(3, random_state=SEED)
for r in p.itertuples():
    print(f"{r.id}  (WER {r.wer_clip}%)\n  real  : {r.text_norm}\n  modelo: {r.prediccion}")
    ipd.display(ipd.Audio(str(RAIZ / r.file_name)))